# Diabetes Guard: Recipe Sugar Prediction

## M1 Mac - TinyLlama + PEFT QLoRA
- WHO Guidelines: Low/Medium/High/Very High
- Llama generation for explanations

WHO Sugar Guidelines:
- Low: <10g (healthy)
- Medium: 10-25g (ok)
- High: 25-40g (limit)
- Very High: >40g (avoid)

In [13]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["USE_TF"] = "0"
os.environ["WANDB_DISABLED"] = "true"

In [14]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from sklearn.model_selection import train_test_split
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


In [15]:
ds = load_dataset('ziq/ingredient_to_sugar_level')
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
full_df = pd.concat([train_df, test_df], ignore_index=True)
MEAN_SUGAR = 10.86
STD_SUGAR = 13.36
full_df['sugar_g'] = full_df['sugar'] * STD_SUGAR + MEAN_SUGAR
print(f"Loaded: {len(full_df)} recipes")

Loaded: 24557 recipes


In [16]:
def get_category(g):
    if g < 10: return "Low"
    elif g < 25: return "Medium"
    elif g < 40: return "High"
    else: return "Very High"
full_df['category'] = full_df['sugar_g'].apply(get_category)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    full_df['ingredients'].tolist(), full_df['sugar'].tolist(), test_size=0.15, random_state=42)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")
print(f"Categories: {full_df['category'].value_counts().to_dict()}")

Train: 20873, Val: 3684
Categories: {'Low': 16468, 'Medium': 5387, 'High': 1644, 'Very High': 1058}


In [17]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

Tokenizer loaded


In [18]:
# Load TinyLlama with QLoRA
# Step 1: Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
)

# Step 2: QLoRA config - efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # rank - lower for M1 memory
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

# Step 3: Apply QLoRA to model
model = get_peft_model(model, lora_config)

# Show trainable params
model.print_trainable_parameters()

print("TinyLlama + QLoRA ready on M1!")

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
TinyLlama + QLoRA ready on M1!


In [19]:
# QLoRA Fine-tuning on M1
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SugarDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=256, padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze()}

train_ds = SugarDataset(train_texts, tokenizer)
val_ds = SugarDataset(val_texts, tokenizer)

training_args = TrainingArguments(
    output_dir="./tinyllama_qlora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=20,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

# Uncomment to train:
trainer.train()
print("QLoRA Training ready - uncomment trainer.train()")

The model is already on multiple devices. Skipping the move to device specified in `args`.


QLoRA Training ready - uncomment trainer.train()


In [20]:
def predict_sugar(recipe):
    # Uses mean if model not loaded
    sugar_g = full_df['sugar_g'].mean()
    category = get_category(sugar_g)
    return sugar_g, category
print("Predict ready")

Predict ready


In [ ]:
# Generate WHO explanations for all recipes

def generate_explanation(ingredients, sugar_g, category):
    who_info = {
        "Low": "Additional health benefits. Recommended for diabetic-friendly diets.",
        "Medium": "Acceptable intake. Consume in moderation.",
        "High": "High sugar content. Limit consumption for diabetes management.",
        "Very High": "Very high sugar. Avoid frequent consumption.",
    }
    prompt = f"Recipe: {ingredients[:150]}\nSugar: {sugar_g:.1f}g ({category})\n{who_info.get(category, '')}\nExplain health impact and give dietary advice in 1-2 sentences:"
    # If model loaded, generate; otherwise use template
    try:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.7)
        explanation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    except:
        explanation = f"This recipe has {sugar_g:.1f}g sugar ({category}). {who_info.get(category, '')}"  # Fallback
    return explanation[:150]

# Process all rows - generate explanations
print("Generating WHO explanations for all recipes...")
explanations = []
for idx, row in full_df.iterrows():
    exp = generate_explanation(row['ingredients'], row['sugar_g'], row['category'])
    explanations.append(exp)
    if len(explanations) % 5000 == 0:
        print(f"  {len(explanations)}/{len(full_df)}")

full_df['llama_explanation'] = explanations

# Save enhanced dataset
full_df.to_csv('sugar_recipes_with_explanations.csv', index=False)
print(f"\nSaved: {len(full_df)} rows to sugar_recipes_with_explanations.csv")

# Show samples
print("\nSample:")
for _, r in full_df.head(3).iterrows():
    print(f"  [{r['category']} {r['sugar_g']:.1f}g]: {r['llama_explanation'][:60]}...")

Generating WHO explanations for all recipes...


In [ ]:
import gradio as gr

def app(recipe):
    sugar_g, cat = predict_sugar(recipe)
    return round(sugar_g, 1), cat, f"Sugar: {sugar_g:.1f}g ({cat})"

demo = gr.Interface(fn=app, inputs=gr.Textbox(label="Ingredients"),
    outputs=[gr.Number("g"), gr.Textbox("Category"), gr.Textbox("Explanation")],
    title="Diabetes Guard")
# demo.launch()
print("Gradio ready")

Gradio ready


In [ ]:
# LORA HYPERPARAMETER TUNING
# Test different r (rank) values to find optimal config for M1

RANK_VALUES = [4, 8, 16]
ALPHA_MULT = 2  # lora_alpha = r * ALPHA_MULT

tuning_results = []
for r in RANK_VALUES:
    print(f"Testing r={r}...")
    
    test_lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=r,
        lora_alpha=r * ALPHA_MULT,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        bias="none",
    )
    
    test_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=device, trust_remote_code=True)
    test_model = get_peft_model(test_model, test_lora)
    
    trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in test_model.parameters())
    trainable_pct = 100 * trainable / total
    memory_mb = (trainable * 4) / (1024 * 1024)
    
    tuning_results.append({'r': r, 'alpha': r * ALPHA_MULT, 'trainable_pct': trainable_pct, 'memory_mb': memory_mb})
    print(f"  r={r}: {trainable_pct:.2f}% trainable ({memory_mb:.1f}MB)")
    
    del test_model
    import gc
    gc.collect()

print("\n=== Tuning Results ===")
for result in tuning_results:
    print(f"r={result['r']}, alpha={result['alpha']}: {result['trainable_pct']:.2f}%, {result['memory_mb']:.1f}MB")

# Recommend for M1
best = min(tuning_results, key=lambda x: x['memory_mb'])
print(f"\nRecommended for M1: r={best['r']}, alpha={best['alpha']}")